# XGBoost TreeSHAP analysis — off-market Binary BoW

This notebook explains the **saved, fitted XGBoost model** that produced your new off-market benchmark. It **does not retrain or refit** the model.

It uses XGBoost's native TreeSHAP contributions (which handle its pandas categorical splits), wrapped as `shap.Explanation` for SHAP plots. The model predicts centered **log sale price**; this notebook correctly restores the target-centering offset into the SHAP baseline.

**Prerequisites (run once in your project terminal):**

```powershell
git pull origin main
uv add shap
```

### Save the correct May validation model once

Run the following **in `model_run.ipynb` after its validation has completed with `XGBoostRegressor` and `Xpt_spatiotemporal__selected_binary`**. Your May model was trained using earlier sales and is suitable for explaining May 2026 predictions.

```python
from save_model import save_model

may = list(VALIDATION_MONTHS).index(pd.Period("2026-05", freq="M"))
off_market = TRACKS.index("off_market")
model_to_save = models[may][off_market]

assert model_to_save.components[0][0].feature_backend == "xgboost"
assert model_to_save.modelspecs[0][1] == "Xpt_spatiotemporal__selected_binary"
assert len(model_to_save.components) == 1

save_model(
    model_to_save,
    "XGBoost_Binary_OffMarket_May2026",
    metadata={
        "track": "off_market",
        "validation_month": "2026-05",
        "text": "47 selected binary terms",
        "target": "centered log sold price",
    },
)
```

If you saved it previously under this name, either reuse that file or choose a new name (the save helper does not overwrite by default). **Only load model files you trust:** joblib deserialization executes Python pickle data.

Now run this notebook. Its output goes into `figures/shap_xgboost_binary/` and `outputs/shap_xgboost_binary/`. Avoid running on all 10,069 homes initially; start with 500. The May cohort is a diagnostic sample, not an independent model-selection set.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from IPython.display import display

import data
from save_model import load_model
from models.shap_analysis import explain_offmarket_xgboost, global_importance

MODEL_NAME = "XGBoost_Binary_OffMarket_May2026"
MONTH = "2026-05"
SAMPLE_SIZE = 500
SEED = 42
FIG_DIR = Path("figures/shap_xgboost_binary")
OUT_DIR = Path("outputs/shap_xgboost_binary")
FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("SHAP version:", shap.__version__)

In [ ]:
model = load_model(MODEL_NAME)
assert model._fitted
assert len(model.components) == 1
assert model.components[0][0].feature_backend == "xgboost"
assert model.modelspecs[0][1] == "Xpt_spatiotemporal__selected_binary"
print("Loaded:", MODEL_NAME)
print("Estimator:", type(model.components[0][0].estimator).__name__)
print("Feature provider:", type(model.components[0][1]).__name__)
print("Log target offset:", model.target_offset_)

In [ ]:
listings = data.load("listings")
sale_months = pd.to_datetime(listings["date_sold"]).dt.to_period("M")
holdout = listings.loc[sale_months == pd.Period(MONTH, freq="M")]
if holdout.empty:
    raise ValueError(f"No sold homes found in {MONTH}")
holdout = holdout.sample(n=min(SAMPLE_SIZE, len(holdout)), random_state=SEED)
homes = data.hide_answers(holdout, "off_market")

print(f"SHAP explaining {len(homes)} homes sold in {MONTH}.")
print("The model's fitted feature provider is reused without calling fit().")

In [ ]:
explanation, X, predicted_price = explain_offmarket_xgboost(model, homes)
print(f"SHAP contributions: {explanation.values.shape} (homes × features)")
print("Model predictions reconstructed from SHAP values; additivity checked.")

y_true = holdout["sold_price"].to_numpy(dtype=float)
ape = 100.0 * np.abs(predicted_price - y_true) / y_true
diagnostics = pd.DataFrame({
    "ml_num": holdout["ml_num"].to_numpy(),
    "month": MONTH,
    "sold_price": y_true,
    "predicted_price": predicted_price,
    "APE_percent": ape,
    "signed_percent": 100 * (predicted_price - y_true) / y_true,
})
display(diagnostics.head())
print(f"Sample MAPE: {ape.mean():.2f}% | Sample MdAPE: {np.median(ape):.2f}%")
diagnostics.to_csv(OUT_DIR / "sample_predictions.csv", index=False)

In [ ]:
importance = global_importance(explanation, X)
display(importance.head(30))
importance.to_csv(OUT_DIR / "global_importance.csv", index=False)

text_importance = importance.loc[importance["is_text"]].copy()
display(text_importance.head(47))
text_importance.to_csv(OUT_DIR / "text_importance.csv", index=False)
print(f"Text features found: {len(text_importance)}")
print("Mean absolute SHAP measures average influence in log-price units, NOT causal importance.")

In [ ]:
shap.plots.bar(explanation, max_display=25, show=False)
plt.gcf().set_size_inches(10, 7)
plt.tight_layout()
plt.savefig(FIG_DIR / "global_bar.png", dpi=160, bbox_inches="tight")
plt.show()

shap.plots.beeswarm(explanation, max_display=25, show=False)
plt.gcf().set_size_inches(10, 7)
plt.tight_layout()
plt.savefig(FIG_DIR / "global_beeswarm.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
word_positions = [
    i for i, c in enumerate(X.columns)
    if c.startswith(("text_", "tfidf_"))
]
if word_positions:
    word_explanation = explanation[:, word_positions]
    shap.plots.bar(word_explanation, max_display=25, show=False)
    plt.gcf().set_size_inches(10, 7)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "text_bar.png", dpi=160, bbox_inches="tight")
    plt.show()

    shap.plots.beeswarm(word_explanation, max_display=25, show=False)
    plt.gcf().set_size_inches(10, 7)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "text_beeswarm.png", dpi=160, bbox_inches="tight")
    plt.show()

    # Binary word presence and average effect, conditional on word being present.
    rows = []
    for i in word_positions:
        is_present = X.iloc[:, i].to_numpy(dtype=float) > 0
        contrib = explanation.values[:, i]
        rows.append({
            "term": explanation.feature_names[i],
            "present_count": int(is_present.sum()),
            "present_rate": float(is_present.mean()),
            "mean_SHAP_present_log": float(contrib[is_present].mean()) if is_present.any() else np.nan,
            "mean_SHAP_absent_log": float(contrib[~is_present].mean()) if (~is_present).any() else np.nan,
            "mean_abs_SHAP_log": float(np.abs(contrib).mean()),
        })
    word_summary = pd.DataFrame(rows).sort_values(
        "mean_abs_SHAP_log", ascending=False
    )
    display(word_summary.head(25))
    word_summary.to_csv(OUT_DIR / "word_presence_effects.csv", index=False)

    print("The original 47 words were chosen by TF-IDF/Lasso.")
    print("These values describe the fitted BINARY booster, not whether the words are causally important.")

In [ ]:
FEATURE_TO_PLOT = "comp_log_price"  # Try square_feet, comp_local_ppsf_median, etc.
if FEATURE_TO_PLOT in list(explanation.feature_names):
    shap.plots.scatter(
        explanation[:, list(explanation.feature_names).index(FEATURE_TO_PLOT)],
        show=False,
    )
    plt.gcf().set_size_inches(8, 5)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "dependence_comp_log_price.png", dpi=160, bbox_inches="tight")
    plt.show()
else:
    print("Feature not found:", FEATURE_TO_PLOT)

print("Categorical codes in beeswarm colors are display-only, not ordinal values.")

In [ ]:
# Examine an unusual error, or replace ROW_NUMBER with any row between 0 and len(homes)-1.
ROW_NUMBER = int(np.argmax(ape))
print("Selected property:")
display(diagnostics.iloc[[ROW_NUMBER]])

shap.plots.waterfall(explanation[ROW_NUMBER], max_display=20, show=False)
plt.gcf().set_size_inches(10, 8)
plt.tight_layout()
plt.savefig(FIG_DIR / "worst_error_waterfall.png", dpi=160, bbox_inches="tight")
plt.show()

local_effects = pd.DataFrame({
    "feature": list(explanation.feature_names),
    "feature_value": [str(v) for v in X.iloc[ROW_NUMBER]],
    "SHAP_log": explanation.values[ROW_NUMBER],
    "price_multiplier": np.exp(explanation.values[ROW_NUMBER]),
})
local_effects = local_effects.reindex(
    local_effects["SHAP_log"].abs().sort_values(ascending=False).index
)
display(local_effects.head(25))
local_effects.to_csv(OUT_DIR / "selected_property_contributions.csv", index=False)

baseline_dollars = float(np.exp(explanation.base_values[ROW_NUMBER]))
print(f"SHAP baseline: ${baseline_dollars:,.0f}")
print(f"Reconstructed predicted sale price: ${predicted_price[ROW_NUMBER]:,.0f}")
print("A SHAP contribution of +0.10 means about +10.5% multiplicatively, not +$0.10.")

## How to interpret the results

- **Global bar:** Mean absolute SHAP, ranking features by influence on *this saved model's log-price predictions*, not necessarily by their unique information value.
- **Beeswarm:** Horizontal position is contribution to log prediction. Red/blue show high/low numeric feature values. Native categorical features are given arbitrary integer codes **for the plot only**, so don't read category color as ordered.
- **Text-only charts:** Which of the 47 binary remarks terms the fitted XGBoost actually uses, and whether their *presence* generally pushes predictions up or down.
- **Dependence:** How the feature value and its SHAP contribution vary over properties. Correlated property/comparable features may share credit.
- **Waterfall:** One property's model prediction decomposed on the log scale. The baseline incorporates your centered-target offset. No causal meaning should be inferred from word contributions.
- **Important limitations:** The five to six months you've repeatedly validated on are not untouched holdouts. SHAP explains this **trained May model** and 500 sampled May predictions; it does not demonstrate model generalization, establish that a term truly matters to housing prices, or by itself identify optimal new feature subsets.